### In this notebook, we are going to explore Retrieval Augmented Generation to enable LLMs to search and answer based on user provided data.
There are broadly 3 steps in RAG
1. Embed - Store the user provided data in a vector DB (chromaDB being used here) by using an embedding function
2. Retrieval - Based on the query, search the relevant part of the data using the same embeddings
3. Augmented Generation - Use the LLM to refine this retrieved information and provide that as a response

#### NOTE: I have not explored chunking strategies in this notebook, there will be a separate notebook for that

#### Import all necessary packages

In [7]:
from google import genai
from google.genai import types
from IPython.display import Markdown

In [8]:
import os

#### Gemini embeddings

In [10]:
with open("../../gemini_key.txt","r") as f:
    string = f.read()

GOOGLE_API_KEY = string

In [11]:
client = genai.Client(api_key=GOOGLE_API_KEY)

In [12]:
for m in client.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp


In [13]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry

from google.genai import types

class GeminiEmbeddingFunction(EmbeddingFunction):
    # Specify whether to generate embeddings for documents, or queries
    document_mode = True
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model="models/text-embedding-004", ## Using text embedding
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]

##### Break down the document to pages

In [14]:
import pdfplumber

def extract_text_from_pdf(pdf_file):
    documents = []
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            documents.append(page.extract_text())
    return documents

In [15]:
DATA_PATH = "C:/Users/LENOVO/Downloads/Z7IIZ6II_TG_Setting.pdf"

In [16]:
documents = extract_text_from_pdf(DATA_PATH)

In [10]:
db.count()
# You can peek at the data too.
# db.peek(1)

72

In [ ]:
##### Store the document in chromaDB

In [ ]:
import chromadb

DB_NAME = "nikon_z72"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

## Retrieval: Find relevant documents

To search the Chroma database, call the `query` method. Note that you also switch to the `retrieval_query` mode of embedding generation.


In [26]:
# Switch to query mode when generating embeddings.
embed_fn.document_mode = False

# Search the Chroma DB using the specified query.
query = "How to set for landscape photography?"

result = db.query(query_texts=[query], n_results=1)
[all_passages] = result["documents"]

Markdown(all_passages[0])

Basic Settings for Landscape Photography
The following are the recommended settings for photo-
graphing landscapes:
PPhhoottoo sshhoooottiinngg mmeennuu
Focus mode Single AF
AF-area mode Single-point AF
Diffraction compensation On
Vibration reduction • Hand-held photography: Normal
• Tripod photography: Off 1
Silent photography On
CCuussttoomm SSeettttiinnggss mmeennuu
a6 AF activation Shutter/AF-ON
a10 Low-light AF On
d4 Exposure delay mode 1 s 2
OOtthheerr sseettttiinnggss
Release mode Single frame
Monitor mode Monitor only
1. Settings for lenses with on-board vibration reduction vary with
the lens (ppaaggee 88).
2. We recommend that, where available, an optional remote
control be used in place of [Exposure delay mode].
6

## Augmented generation: Answer the question

Now that you have found a relevant passage from the set of documents (the *retrieval* step), you can now assemble a generation prompt to have the Gemini API *generate* a final answer. Note that in this example only a single passage was retrieved. In practice, especially when the size of your underlying data is large, you will want to retrieve more than one result and let the Gemini model determine what passages are relevant in answering the question. For this reason it's OK if some retrieved passages are not directly related to the question - this generation step should ignore them.

In [27]:
query_oneline = query.replace("\n", " ")

# This prompt is where you can specify any guidance on tone, or what topics the model should stick to, or avoid.
prompt = f"""You are a helpful and informative bot that answers questions using text from the reference passage included below. 
Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. 
You are talking to a technical audience, so be sure to break down complicated concepts and 
strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: {query_oneline}
"""

# Add the retrieved documents to the prompt.
for passage in all_passages:
    passage_oneline = passage.replace("\n", " ")
    prompt += f"PASSAGE: {passage_oneline}\n"

print(prompt)

You are a helpful and informative bot that answers questions using text from the reference passage included below. 
Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. 
You are talking to a technical audience, so be sure to break down complicated concepts and 
strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: How to set for landscape photography?
PASSAGE: Basic Settings for Landscape Photography The following are the recommended settings for photo- graphing landscapes: PPhhoottoo sshhoooottiinngg mmeennuu Focus mode Single AF AF-area mode Single-point AF Diffraction compensation On Vibration reduction • Hand-held photography: Normal • Tripod photography: Off 1 Silent photography On CCuussttoomm SSeettttiinnggss mmeennuu a6 AF activation Shutter/AF-ON a10 Low-light AF On d4 Exposure delay mode 1 s 2 OOtthheerr sseettttiinnggss Release mode Single frame Monitor mod

Now use the `generate_content` method to to generate an answer to the question.

In [28]:
answer = client.models.generate_content(
    model="gemini-2.0-flash",
    contents=prompt)

Markdown(answer.text)

To configure your camera for landscape photography, the recommended settings are to set the focus mode to Single AF and the AF-area mode to Single-point AF. It's also advisable to turn diffraction compensation on. For vibration reduction, use Normal mode for handheld photography and turn it off when using a tripod. Enable Silent photography, set AF activation to Shutter/AF-ON, turn on Low-light AF, and set Exposure delay mode to 1 second. Lastly, use Single frame release mode and set the monitor mode to Monitor only. Remember that settings for lenses with on-board vibration reduction may vary, and using a remote control instead of Exposure delay mode is recommended where available.
